# Base de jugadores de Primera Nacional (todas las temporadas)

Junta en `data/jugadores/` a **todos** los jugadores de la categoría, temporada por temporada, a partir de lo que ya está scrapeado en `data/temporadas/{año}/` (Chaco + todos los rivales). No scrapea nada.

| Archivo | Una fila por |
|---|---|
| `partidos.csv` | partido (`event_id`), con DT de cada equipo y qué datos trae |
| `equipos.csv` | equipo y temporada, con zona y posición final |
| `jugadores.csv` | persona (`player_id`): bio, equipos, minutos de toda su historia |
| `jugador_partido/{año}.csv` | jugador y partido, con **todas** las stats de SofaScore (también los suplentes que no entraron) |
| `jugador_temporada.csv` | jugador, temporada y equipo: el mismo pool de `scouting.py` + extras |

Reglas que conviene tener presentes:
- Cada partido está guardado en la carpeta de los dos equipos: se usa **una** copia, la más completa y, a igualdad, la más nueva (SofaScore corrige datos después del partido).
- `nivel_datos`: en un partido con análisis la stat vacía vale 0; en uno sin análisis (`basico`) queda vacía.
- La edad de `jugador_temporada` es al 1 de julio de esa temporada.
- Es incremental: solo vuelve a leer los partidos nuevos o re-scrapeados.

In [ ]:
import os, sys
%load_ext autoreload
%autoreload 2
if os.path.basename(os.getcwd()) in ('primera_nacional', 'liga_profesional'):
    os.chdir('..')
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
PROJECT_ROOT = os.getcwd(); sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import config
from src.jugadores import construir as C
from IPython.display import display

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

## 1. Configuración

In [ ]:
# ============================================================
# CONFIGURAR ACÁ
TEMPORADAS = None    # None = todas las de data/temporadas/; o ['2024'] para una sola
FORZAR     = False   # True para releer TODOS los partidos aunque no hayan cambiado
# ============================================================

## 2. Construir / actualizar la base

In [ ]:
C.construir(TEMPORADAS, forzar=FORZAR)

## 3. Controles

- `verificar`: un partido por `event_id`, ningún jugador con más partidos que fechas y ~11 en cancha por equipo (si diera 22 sería doble conteo).
- `comparar_con_pool`: arma la temporada con las mismas copias que `scouting.py` y la compara contra `pool_liga()`. Lo esperado es que solo difieran `edad` (al 1/7 contra hoy) y `equipo` (la base usa el nombre de config; el pool mezcla grafías como "Ciudad de Bolivar"/"Bolívar"). Tarda ~1 minuto.

In [ ]:
partidos = C.cargar('partidos')
for t in sorted(partidos['temporada'].astype(str).unique()):
    C.verificar(t)

In [ ]:
C.comparar_con_pool('2026');

## 4. Jugadores con dos `player_id`

SofaScore a veces duplica o fusiona perfiles, y un mismo jugador queda con dos ids. Esto lista los candidatos (mismo nombre, mismo equipo y temporada, nunca juntos en cancha). **No une nada solo**: si es la misma persona (mismo nombre completo y nacimiento compatible), cargalo en `config.ALIAS_JUGADORES = {id_duplicado: id_real}` y volvé a correr la sección 2.

In [ ]:
dups = C.posibles_duplicados()

## 5. Régimen de los datos (cambio de medición del 1/7/2026)

SofaScore cambió cómo mide la Primera Nacional desde el 1/7/2026 (la Liga Profesional no): quites -55%, regates -35%, duelos perdidos -27%, despejes +77%, ocasiones claras +40%, xG +21%, y los malos controles dejaron de venir. 2025 y el primer semestre 2026 son comparables entre sí.

Cada partido lleva su `regimen` (A hasta el 30/6/2026, B desde el 1/7) y acá se calculan los factores por estadística y posición que llevan el B a la escala del A. Validado con los 367 jugadores que jugaron los dos semestres en el mismo equipo: en crudo sus quites "caen" 51% de un semestre al otro; ajustado, 0%.

In [ ]:
from src.jugadores import consultas as Q
factores = Q.factores_regimen()

## 6. Pool de todas las temporadas y percentiles

- `Q.pool(...)`: una fila por (temporada, jugador, equipo). `ajustado=True` (default) aplica los factores de régimen; `ajustado=False` es el pool crudo, igual al de `scouting.py`.
- `Q.percentiles(pool, universo='temporada')`: contra los de su rol en la misma temporada. `universo='historico'`: contra todas las temporadas juntas (más muestra).
- En las métricas donde menos es mejor (pérdidas, faltas, errores...) el percentil está invertido: **alto = bueno**.
- Cada métrica trae `_pct`, `_puesto` y `_de` (para el "3° de 111").

In [ ]:
# ============================================================
MIN_MINUTOS_UNIVERSO = 600
# ============================================================
pool = Q.pool()
pct_temporada = Q.percentiles(pool, universo='temporada', min_minutos=MIN_MINUTOS_UNIVERSO)
pct_historico = Q.percentiles(pool, universo='historico', min_minutos=MIN_MINUTOS_UNIVERSO)
print(len(pool), 'filas jugador-temporada ·', pool['temporada'].value_counts().sort_index().to_dict())

## 7. Buscar un jugador: recorrido y puestos

In [ ]:
# ============================================================
NOMBRE   = 'Pacheco'   # texto del nombre corto o completo (sin importar tildes)
METRICAS = ['xg_p90', 'npxg_p90', 'key_passes_p90', 'dribbles_won_p90', 'tackles_p90', 'rating']
# ============================================================
encontrados = Q.buscar(NOMBRE)
display(encontrados[['player_id', 'nombre_corto', 'nombre_completo', 'nacimiento', 'posicion_habitual',
                     'equipos', 'primera_temporada', 'ultima_temporada', 'partidos_total', 'minutos_total']])
if not encontrados.empty:
    pid = encontrados.iloc[0]['player_id']
    bio, tabla, carrera = Q.historial(pid)
    display(tabla)
    print('Carrera en la categoría:', {k: v for k, v in carrera.items() if not k.endswith('_p90')})
    for t in tabla['temporada']:
        for m in METRICAS:
            a = Q.puesto(pct_temporada, pid, m, temporada=t)
            b = Q.puesto(pct_historico, pid, m, temporada=t)
            if a and b:
                print(f"{t} · {m:18s} {a['valor']:.2f}  →  {a['texto']}  |  {b['texto']}")

## 8. Comparables de todas las temporadas (gemelos k-NN)

Mismo motor que `scouting_similitud` (features del rol, suavizado, escalado y pesos del score), pero buscando en **todas** las temporadas de la base y sobre el pool ajustado. Cada candidato es "cómo jugaba X en tal temporada". El jugador no aparece como gemelo de sí mismo.

Filtros: `edad_max`, `edad_min`, `minutos_min`, `pj_min`, `valor_max_eur`, `excluir_equipos`, `solo_equipos`, `banda`, y `temporadas_candidatos` (ej. `['2026']` para quedarte con los que están jugando ahora).

In [ ]:
# ============================================================
PLAYER_ID = encontrados.iloc[0]['player_id'] if not encontrados.empty else None   # o un id a mano
TEMPORADA_REF = None          # None = su temporada con más minutos
FILTROS_COMP = {'edad_max': 28, 'minutos_min': 900}
# ============================================================
if PLAYER_ID is not None:
    comp = Q.comparables(PLAYER_ID, temporada=TEMPORADA_REF, k=12, filtros=FILTROS_COMP)
    print('Referencia:', comp.attrs['referencia'])
    display(comp)

## 9. Chequeo de la medición de SofaScore (correr al cerrar cada temporada)

Promedio por equipo y partido de las stats más sensibles, mes a mes, dentro de cada régimen. Si aparece un salto **sostenido** de varios meses (como el de julio 2026), es un cambio de medición: agregar el corte en `construir.CORTES_REGIMEN` y volver a correr la sección 2. Un mes suelto con pocos partidos suele ser ruido.

In [ ]:
mensual, alertas = Q.chequeo_medicion()

## Al sumar una temporada vieja

1. Registrar en `config.py` los equipos de ese año que todavía no estén (`RIVALS_<año>` con su `team_id`), para que el slug quede fijo.
2. Agregar el `season_id` de SofaScore en `config.SEASON_IDS`.
3. Scrapear (Chaco + rivales) y correr este notebook: solo lee lo nuevo.
4. Revisar la sección 4 (duplicados): los que traen `evidencia = shotmap ...` son casi seguros (mismo partido, mismo equipo, mismo nombre).
5. Si SofaScore vuelve a cambiar la medición, agregar el corte en `construir.CORTES_REGIMEN`.